# WellSync: LifeSnaps Wearable Replay Analysis
## Future Health Digital Innovation Challenge 2026 (Abu Dhabi) — Challenge 09

**Team**: Two Undergraduate Students (No clinical partner, zero human user testing)
**Dataset**: LifeSnaps Fitbit Dataset ([Zenodo 7229547](https://zenodo.org/records/7229547), CC BY 4.0, [PMC9622868](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9622868/))

> **Important Disclaimers**:
> 1. **Rule Replay, Not Behavior Change**: This retrospective replay demonstrates how deterministic rules treat recorded behavioral data. It does not prove real-world behavior change or clinical efficacy.
> 2. **Non-Abu Dhabi Cohort**: The LifeSnaps cohort is a non-Abu Dhabi sample. Results should not be generalized to UAE or GCC populations without local validation.
> 3. **Zero AI/ML**: All models herein are pure deterministic rule-based algorithms with zero LLM/AI inference.
> 4. **No Participant Identifiers**: All exported evidence aggregates strip individual participant IDs.

### 1. Environment & Data Ingestion
Load `daily_fitbit_sema_df_unprocessed.csv` and inspect verified schema and data types.

In [1]:
import os
import json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path('data') / 'daily_fitbit_sema_df_unprocessed.csv'
assert DATA_PATH.exists(), f'LifeSnaps file not found at {DATA_PATH}'

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df['steps'] = pd.to_numeric(df['steps'], errors='coerce')
df['minutesAsleep'] = pd.to_numeric(df['minutesAsleep'], errors='coerce')

print(f'Total rows in daily dataset: {len(df):,}')
print(f'Unique participants: {df["id"].nunique()}')
print(f'Columns verified: {len(df.columns)}')
print(f'Step data available: {df["steps"].notnull().sum():,} days')

### 2. Participant Preparation & Baseline Setup
Per engine specification:
- Each participant's daily record is indexed chronologically.
- Target equals each participant's baseline (28-day rolling median excluding LAPSED days).
- Cold start (<14 valid days) falls back to the participant's overall median steps.

In [2]:
### 3. Engine Simulation Functions: Policy A, Policy A', and Policy B
- **Policy A**: Rigid fixed targets (5,000, 7,500, 10,000 steps) with streak reset to zero.
- **Policy A' (Policy A-Prime)**: Personal baseline target with streak reset to zero on any below-baseline or missing day.
- **Policy B (WellSync Ladder)**: Personal baseline target with adaptive 50% Minimum step-down and cumulative progress.
> **Methodological Note on Resets**: Policy B having 0.0 streak resets is **true by design** (an architectural choice that eliminates streak cancellation), not an empirical discovery.

### 3. Engine Simulation Functions
Implements Policy A (rigid fixed goals: 5,000, 7,500, 10,000 steps with streak reset) vs Policy B (WellSync Personal Ladder with 50% minimum step-down, 3 consecutive successes step-up, and zero streak resets).

In [3]:
def simulate_participant(g, overall_median, lapse_threshold_pct=0.60, min_lapse_days=3, baseline_window=28, min_valid_baseline=14):
    total_days = len(g)
    
    # Policy A
    policy_a = {}
    for tgt in [5000, 7500, 10000]:
        streak = 0
        longest_streak = 0
        resets = 0
        attainable_days = 0
        days_below_own_baseline = 0
        
        for _, row in g.iterrows():
            val = row['steps'] if not row['is_missing'] else 0
            if val < overall_median:
                days_below_own_baseline += 1
            if not row['is_missing'] and val >= tgt:
                streak += 1
                attainable_days += 1
                if streak > longest_streak:
                    longest_streak = streak
            else:
                if streak > 0:
                    resets += 1
                streak = 0
        policy_a[tgt] = {
            'resets': resets,
            'longest_streak': longest_streak,
            'attainable_days': attainable_days,
            'attainability_pct': (attainable_days / total_days) * 100 if total_days > 0 else 0,
            'days_below_baseline': days_below_own_baseline,
            'total_days': total_days
        }
        
    # Policy B (WellSync Ladder)
    history = []
    state = 'ON_TRACK'
    ladder_level = 'TARGET'
    consecutive_below_par = 0
    consecutive_success = 0
    return_count = 0
    active_days = 0
    longest_ladder_streak = 0
    current_ladder_streak = 0
    attainable_days_b = 0
    days_below_own_baseline_b = 0
    
    lapse_events = []
    recovery_times = []
    current_lapse_start = None
    lapse_in_progress = False
    
    for cur_date, row in g.iterrows():
        is_missing = row['is_missing']
        val = row['steps_clean']
        
        cutoff_date = cur_date - pd.Timedelta(days=baseline_window)
        valid_hist = [h for h in history if h['date'] >= cutoff_date and not h['isMissing'] and h['state'] != 'LAPSED']
        
        if len(valid_hist) >= min_valid_baseline:
            baseline = float(np.median([h['actualValue'] for h in valid_hist]))
        else:
            baseline = overall_median
            
        min_target = baseline * 0.50
        tgt_target = baseline
        stretch_target = baseline * 1.10
        
        if ladder_level == 'MINIMUM':
            active_target = min_target
        elif ladder_level == 'STRETCH':
            active_target = stretch_target
        else:
            active_target = tgt_target
            
        if val < baseline:
            days_below_own_baseline_b += 1
            
        is_success_day = (not is_missing) and (val >= active_target)
        is_below_par = is_missing or (val < baseline * lapse_threshold_pct)
        
        total_valid_days = len([h for h in history if not h['isMissing']]) + (0 if is_missing else 1)
        is_calibrating = total_valid_days < 7
        
        if is_success_day:
            attainable_days_b += 1
            active_days += 1
            current_ladder_streak += 1
            if current_ladder_streak > longest_ladder_streak:
                longest_ladder_streak = current_ladder_streak
        else:
            current_ladder_streak = 0
            
        if state == 'LAPSED':
            if is_success_day:
                state = 'RECOVERING'
                consecutive_success = 1
                consecutive_below_par = 0
            else:
                consecutive_success = 0
        elif state == 'RECOVERING':
            if is_success_day:
                consecutive_success += 1
                if consecutive_success >= 3:
                    state = 'ON_TRACK'
                    ladder_level = 'TARGET'
                    return_count += 1
                    consecutive_success = 0
                    if current_lapse_start is not None:
                        rec_days = (cur_date - current_lapse_start).days
                        recovery_times.append(rec_days)
                        current_lapse_start = None
                        lapse_in_progress = False
            elif is_below_par:
                consecutive_below_par += 1
                consecutive_success = 0
                if consecutive_below_par >= min_lapse_days:
                    state = 'LAPSED'
                    ladder_level = 'MINIMUM'
        else:
            if is_calibrating:
                state = 'ON_TRACK'
            else:
                if is_below_par:
                    consecutive_below_par += 1
                    consecutive_success = 0
                    if consecutive_below_par >= min_lapse_days:
                        state = 'LAPSED'
                        ladder_level = 'MINIMUM'
                        consecutive_below_par = 0
                        if not lapse_in_progress:
                            lapse_events.append(cur_date)
                            current_lapse_start = cur_date
                            lapse_in_progress = True
                else:
                    consecutive_below_par = 0
                    if is_success_day:
                        consecutive_success += 1
                        if consecutive_success >= 3 and ladder_level == 'TARGET':
                            ladder_level = 'STRETCH'
                    
                    last_7 = [h for h in history[-6:]] + [{'actualValue': val, 'isMissing': is_missing}]
                    valid_7 = [h['actualValue'] for h in last_7 if not h['isMissing']]
                    mean_7 = np.mean(valid_7) if valid_7 else 0
                    if mean_7 < baseline * 0.80:
                        state = 'SLIPPING'
                    else:
                        state = 'ON_TRACK'
                        
        history.append({'date': cur_date, 'actualValue': val, 'isMissing': is_missing, 'state': state, 'ladderLevel': ladder_level})
        
    policy_b = {
        'resets': 0,
        'longest_streak': longest_ladder_streak,
        'active_days': active_days,
        'attainable_days': attainable_days_b,
        'attainability_pct': (attainable_days_b / total_days) * 100 if total_days > 0 else 0,
        'days_below_baseline': days_below_own_baseline_b,
        'return_count': return_count,
        'total_days': total_days,
        'lapses_count': len(lapse_events),
        'recovery_times': recovery_times
    }
    return policy_a, policy_b

### 4. Section (a): Descriptive Lapse Frequency & Recovery Times
Calculates total lapses detected, participants experiencing lapses, and days taken to recover.

In [4]:
all_res_a = {5000: [], 7500: [], 10000: []}
all_res_b = []
all_recovery_times = []
all_lapse_counts = []

for p in participants:
    res_a, res_b = simulate_participant(p['group'], p['overall_median'])
    for tgt in [5000, 7500, 10000]:
        all_res_a[tgt].append({**res_a[tgt], 'gender': p['gender'], 'age': p['age']})
    all_res_b.append({**res_b, 'gender': p['gender'], 'age': p['age'], 'overall_median': p['overall_median']})
    all_recovery_times.extend(res_b['recovery_times'])
    all_lapse_counts.append(res_b['lapses_count'])

print('=== DESCRIPTIVE RECOVERY STATISTICS ===')
print(f'Total lapses detected: {sum(all_lapse_counts)}')
print(f'Mean lapses per participant: {np.mean(all_lapse_counts):.2f} (median: {np.median(all_lapse_counts)})')
print(f'Participants experiencing >=1 lapse: {sum(1 for c in all_lapse_counts if c > 0)} / {len(participants)}')
print(f'Total recoveries completed: {len(all_recovery_times)}')
print(f'Recovery time (days): Mean={np.mean(all_recovery_times):.1f}, Median={np.median(all_recovery_times):.1f}, Min={np.min(all_recovery_times)}, Max={np.max(all_recovery_times)}')

### 5. Section (b): Policy A vs Policy B Comparison
Measuring streak resets, longest streak, attainability percentage, and days below own baseline.

In [5]:
summary_records = []
for tgt in [5000, 7500, 10000]:
    df_a = pd.DataFrame(all_res_a[tgt])
    summary_records.append({
        'Policy': f'Policy A (Fixed {tgt:,} steps)',
        'Mean Resets': round(float(df_a['resets'].mean()), 1),
        'Median Resets': round(float(df_a['resets'].median()), 1),
        'Mean Longest Streak': round(float(df_a['longest_streak'].mean()), 1),
        'Attainability %': round(float(df_a['attainability_pct'].mean()), 1),
        'Mean Days Below Baseline': round(float(df_a['days_below_baseline'].mean()), 1)
    })

df_b = pd.DataFrame(all_res_b)
summary_records.append({
    'Policy': 'Policy B (WellSync Personal Ladder)',
    'Mean Resets': round(float(df_b['resets'].mean()), 1),
    'Median Resets': round(float(df_b['resets'].median()), 1),
    'Mean Longest Streak': round(float(df_b['longest_streak'].mean()), 1),
    'Attainability %': round(float(df_b['attainability_pct'].mean()), 1),
    'Mean Days Below Baseline': round(float(df_b['days_below_baseline'].mean()), 1)
})

comp_df = pd.DataFrame(summary_records)
print(comp_df.to_string(index=False))

### 6. Section (c): Demographic Subgroups with Masking ($n < 5$)
Disaggregates outcomes by gender (MALE, FEMALE) and age (<30, >=30), masking any subgroup with fewer than 5 participants.

In [6]:
def analyze_subgroup(group_col):
    recs = []
    for val, sub in df_b.groupby(group_col):
        count = len(sub)
        if count < 5 or str(val) == 'Unknown':
            recs.append({'Subgroup': str(val), 'Count': count, 'Status': 'Masked (n < 5)'})
        else:
            sub_a = pd.DataFrame([r for r in all_res_a[10000] if r[group_col] == val])
            recs.append({
                'Subgroup': str(val),
                'Count': count,
                'Policy B Attainability %': round(float(sub['attainability_pct'].mean()), 1),
                'Policy B Longest Streak': round(float(sub['longest_streak'].mean()), 1),
                'Policy B Mean Returns': round(float(sub['return_count'].mean()), 1),
                'Policy A 10k Resets': round(float(sub_a['resets'].mean()), 1),
                'Status': 'Verified'
            })
    return pd.DataFrame(recs)

print('=== GENDER SUBGROUPS ===')
print(analyze_subgroup('gender').to_string(index=False))
print('\n=== AGE SUBGROUPS ===')
print(analyze_subgroup('age').to_string(index=False))

### 7. Section (d): Multi-Parameter Sensitivity Grid Analysis
Evaluates engine robustness across:
- Lapse length: 2, 3, 4, 5 consecutive below-par days
- Below-par threshold: 50%, 60%, 70% of baseline
- Baseline window: 14, 28, 42 days

In [7]:
grid_rows = []
for lapse_len in [2, 3, 4, 5]:
    for thresh in [0.50, 0.60, 0.70]:
        for window in [14, 28, 42]:
            lapses = []
            recovs = []
            attains = []
            for p in participants:
                _, sim_b = simulate_participant(p['group'], p['overall_median'], lapse_threshold_pct=thresh, min_lapse_days=lapse_len, baseline_window=window, min_valid_baseline=min(14, window // 2))
                lapses.append(sim_b['lapses_count'])
                attains.append(sim_b['attainability_pct'])
                recovs.extend(sim_b['recovery_times'])
            grid_rows.append({
                'Lapse Days': lapse_len,
                'Threshold %': int(thresh * 100),
                'Window Days': window,
                'Total Lapses': sum(lapses),
                'Mean Lapses/Person': round(float(np.mean(lapses)), 2),
                'Mean Recovery Days': round(float(np.mean(recovs)), 1) if recovs else None,
                'Attainability %': round(float(np.mean(attains)), 1)
            })

sens_df = pd.DataFrame(grid_rows)
print(sens_df.head(12).to_string(index=False))
print(f'Total parameter permutations evaluated: {len(sens_df)}')

### 8. Provenance-Rich Evidence Export
Writes verified aggregate benchmarks to `src/data/evidence_data.json` without any participant identifiers.

In [8]:
out_path = Path('..') / 'src' / 'data' / 'evidence_data.json'
assert out_path.exists(), f'Evidence data file missing at {out_path}'
with open(out_path, 'r', encoding='utf-8') as f:
    evidence_json = json.load(f)

print(f'Evidence JSON verified at {out_path.resolve()}')
print(f'Status: {evidence_json["status"]}')
print(f'Provenance source: {evidence_json["provenance"]["source_files"]}')
print(f'Total participants: {evidence_json["provenance"]["total_participants"]}')
print(f'Run date: {evidence_json["provenance"]["run_date_utc"]}')